In [ ]:
# @title 🚀 शिव एआई - मास्टर इंटीग्रेशन और गिटहब सिंक
import os
from google.colab import drive, output
from transformers import AutoModelForCausalLM, AutoTokenizer
from IPython.display import display, HTML, Audio
from gtts import gTTS
import torch, ipywidgets as widgets

# १. ड्राइव और ओनर की जानकारी
OWNER = "Shri Ram nag"
DRIVE_PATH = "/content/drive/MyDrive/ShivAI_Backup"

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

print("⌛ शिव एआई का दिमाग लोड हो रहा है...")
try:
    tokenizer = AutoTokenizer.from_pretrained(DRIVE_PATH)
    model = AutoModelForCausalLM.from_pretrained(DRIVE_PATH, torch_dtype=torch.bfloat16, device_map="auto")
    print("✅ १००० स्टेप्स मॉडल लोड हो गया है!")
except:
    print("❌ मॉडल नहीं मिला! कृपया बैकअप फोल्डर चेक करें।")

In [ ]:
# @title 🧠 शिव एआई - क्लीन वॉइस और चैट फिक्स
def speak_now(text):
    # हकलाहट रोकने के लिए नंबर को शब्दों में बदलें
    n_map = {"1":"एक", "2":"दो", "3":"तीन", "4":"चार", "5":"पाँच", "0":"शून्य", ".":"डॉट"}
    for n, w in n_map.items():
        text = text.replace(n, w)
    tts = gTTS(text, lang='hi')
    filename = "shiv_voice.wav"
    tts.save(filename)
    return filename

def ai_logic_clean(u_in):
    sys = f"You are Shiv AI by {OWNER}. Answer shortly in Hindi."
    prompt = f"<|system|>\n{sys}<|user|>\n{u_in}<|assistant|>\n"
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    # वॉर्निंग और रिपीटेशन रोकने के लिए पैरामीटर्स
    ids = model.generate(
        **inputs, 
        max_new_tokens=60, 
        repetition_penalty=1.3, 
        temperature=0.3, 
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    
    raw_res = tokenizer.decode(ids[0], skip_special_tokens=True)
    return raw_res.split("<|assistant|>")[-1].strip()

ui_html = HTML(f'''
<div style="background:#121212; color:#FF4500; padding:20px; border-radius:15px; border:3px solid #FF4500; text-align:center;">
    <h2>🚩 शिव एआई (Shiv AI)</h2>
    <p>मालिक: {OWNER}</p>
    <button id="m-btn" style="width:60px; height:60px; border-radius:50%; background:#2196F3; color:white; font-size:25px; cursor:pointer;" onclick="startMic()">🎤</button>
    <p id="st">तैयार हूँ...</p>
</div>
<script>
    function startMic() {{
        var r = new (window.SpeechRecognition || window.webkitSpeechRecognition)();
        r.lang = 'hi-IN';
        r.onstart = () => {{ document.getElementById('m-btn').style.background='#f44336'; }};
        r.onresult = (e) => {{ google.colab.kernel.invokeFunction('notebook.chat', [e.results[0][0].transcript], {{}}); }};
        r.onend = () => {{ document.getElementById('m-btn').style.background='#2196F3'; }};
        r.start();
    }}
</script>''')

box = widgets.Output()
def chat(msg):
    with box:
        print(f"👤 आप: {msg}")
        ans = ai_logic_clean(msg)
        print(f"🚩 शिव एआई: {ans}")
        display(Audio(speak_now(ans), autoplay=True))

output.register_callback('notebook.chat', chat)
display(ui_html, box)